Create a machine learning model to predict if a bank client is about to cancel the account.

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.utils import shuffle


In [2]:
def lr_model_scores(features_train, target_train, features_val, target_val,
                    weight=None):
    '''
    Returns LogisticRegression model end its scores.
    '''
    model = LogisticRegression(
        solver='liblinear',
        random_state=123,
        class_weight=weight
    )
    model.fit(features_train, target_train)

    score_train = model.score(features_train, target_train)
    score_val = model.score(features_val, target_val)

    predicted_valid = model.predict(features_val)
    f1 = f1_score(target_val, predicted_valid)

    proba_valid = model.predict_proba(features_val)
    proba_one_valid = proba_valid[:, 1]
    roc_auc = roc_auc_score(target_val, proba_one_valid)

    print(
        f'train score: {score_train:.3f}\n'
        f'val score: {score_val:.3f}\n'
        f'F1 Score: {f1:.3f}\n'
        f'ROC AUC: {roc_auc:.3f}'
    )

    return model, f1, roc_auc


def rfc_optimizer(features_train, target_train, features_val, target_val,
                  weight=None):
    '''
    Returns a RandonForestClassifier model with best n_estimators between 
    1 and 31 and best max_depth between 1 and 20, maximizing the F1 Score.
    '''

    best_model = None
    best_n_estimators = 0
    best_depth = 0
    best_f1_score = 0
    for estimators in range(1, 31):
        for depth in range(1, 21):
            model = RandomForestClassifier(
                n_estimators=estimators,
                max_depth=depth,
                random_state=123,
                class_weight=weight
            )
            model.fit(features_train, target_train)

            predicted_valid = model.predict(features_val)
            f1 = f1_score(target_val, predicted_valid)

            if f1 > best_f1_score:
                best_n_estimators = estimators
                best_depth = depth
                best_f1_score = f1
                best_model = model

    score_train = best_model.score(features_train, target_train)
    score_val = best_model.score(features_val, target_val)

    proba_valid = best_model.predict_proba(features_val)
    proba_one_valid = proba_valid[:, 1]
    roc_auc = roc_auc_score(target_val, proba_one_valid)

    print(
        f'best_depth: {best_depth}\n'
        f'best_n_estimators: {best_n_estimators}\n'
        f'train score: {score_train:.3f}\n'
        f'val score: {score_val:.3f}\n'
        f'F1 Score: {best_f1_score:.3f}\n'
        f'ROC AUC: {roc_auc:.3f}'
    )

    return best_model, best_f1_score, roc_auc


def dtc_optimizer(features_train, target_train, features_val, target_val,
                  weight=None):
    '''
    Returns a DecisionTreeClassifier model with best max_depth between 
    1 and 50, maximizing the F1 Score.
    '''
    best_model = None
    best_depth = 0
    best_f1_score = 0
    for depth in range(1, 51):
        model = DecisionTreeClassifier(
            max_depth=depth, 
            random_state=123,
            class_weight=weight
        )

        model.fit(features_train, target_train)
        
        predicted_valid = model.predict(features_val)
        f1 = f1_score(target_val, predicted_valid)
        
        if f1 > best_f1_score:
            best_depth = depth
            best_f1_score = f1
            best_model = model
        
    score_train = best_model.score(features_train, target_train)
    score_val = best_model.score(features_val, target_val)

    proba_valid = best_model.predict_proba(features_val)
    proba_one_valid = proba_valid[:, 1]
    roc_auc = roc_auc_score(target_val, proba_one_valid)
                
    print(
        f'best_depth: {best_depth}\n'
        f'train score: {score_train:.3f}\n'
        f'val score: {score_val:.3f}\n'
        f'F1 Score: {best_f1_score:.3f}\n'
        f'ROC AUC: {roc_auc:.3f}'

    )

    return best_model, best_f1_score, roc_auc


def upsample(features, target):
    '''upsample df by multiplying it where target is 1'''
    # number to multiply the minority class
    multiplier = round(target.value_counts()[0] /
                       target.value_counts()[1])
    
    features_zero = features[target == 0]
    features_one = features[target == 1]
    target_zero = target[target == 0]
    target_one = target[target == 1]

    features_up = pd.concat([features_zero] + [features_one] * multiplier)
    target_up = pd.concat([target_zero] + [target_one] * multiplier)

    features_up, target_up = shuffle(features_up, target_up, random_state=123)

    return features_up, target_up


def downsample(features, target):
    '''downsample df by sampling a fraction where target is 0'''
    fraction = target.value_counts()[1] / target.value_counts()[0]

    features_zero = features[target == 0]
    features_one = features[target == 1]
    target_zero = target[target == 0]
    target_one = target[target == 1]

    features_down = pd.concat(
        [features_zero.sample(frac=fraction, random_state=123)] + 
        [features_one]
    )
    target_down = pd.concat(
        [target_zero.sample(frac=fraction, random_state=123)] +
        [target_one]
    )

    features_down, target_down = shuffle(
        features_down,
        target_down,
        random_state=123
    )

    return features_down, target_down     

In [3]:
# data frame to store the results
scores_df = pd.DataFrame(columns=['model', 'f1', 'auc_roc'])

In [4]:
df = pd.read_csv('./data/Churn.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), str(3)
memory usage: 1.1 MB


In [5]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0


The column 'Exited' shows if the costumer cancelled (1) or not (0). It is the model prediction objective.

## 1 - Cleaning

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df['CustomerId'].duplicated().sum()

np.int64(0)

There is no duplicated row or 'CustomerId'.

The columns 'RowNumber', 'CustomerId' and 'Surname' are not relevant for model training and can be discarded.

In [8]:
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

There are null values in the 'Tenure' column. Verify if they are 0.

In [9]:
df['Tenure'].value_counts(dropna=False)

Tenure
1.0     952
2.0     950
8.0     933
3.0     928
5.0     927
7.0     925
NaN     909
4.0     885
9.0     882
6.0     881
10.0    446
0.0     382
Name: count, dtype: int64

These null are not 0, because there are observations with 0. Verify if there is something in common with these customers.

In [10]:
df[df['Tenure'].isnull()].sample(10, random_state=123)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
7811,611,France,Male,30,NaN,104145.65,1,0,0,159629.64,0
3543,675,France,Male,49,NaN,135133.39,1,0,1,179521.24,1
6471,850,France,Female,40,NaN,0.00,2,1,0,35034.15,0
94,730,Spain,Male,42,NaN,0.00,2,0,1,85982.47,0
6187,704,France,Male,36,NaN,114370.41,1,0,1,66810.48,0
7026,677,France,Female,27,NaN,0.00,2,0,1,114685.92,0
5314,743,Germany,Male,45,NaN,144677.19,3,1,0,22512.44,1
4445,739,France,Male,29,NaN,0.00,2,1,1,164484.78,0
9006,661,Spain,Male,20,NaN,0.00,1,1,0,110252.53,0
1960,655,Spain,Female,36,NaN,135515.76,1,1,0,86013.96,0


There is nothing in common between them. These will be discarded.

In [11]:
df = df.dropna()

## 2 - Preparing

### 2.1 - Class Imbalance Analysis

In [12]:
df['Exited'].value_counts(normalize=True)

Exited
0    0.796062
1    0.203938
Name: proportion, dtype: float64

Only 20% of the values are 1. This will not be balanced now to compare the model before and after considering the imbalance.

### 2.2 - Encode Cathegorical Characteristics

In [13]:
df['Geography'].unique()

<StringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str

In [14]:
df['Gender'].unique()

<StringArray>
['Female', 'Male']
Length: 2, dtype: str

Some models performs better with One-Hot Encoding, others with Label Encoding.

Two dfs will be created, one for each encoding technique.

#### 2.2.1 One-Hot Encoding

In [15]:
df_ohe = pd.get_dummies(df, drop_first=True)
df_ohe.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2.0,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1.0,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8.0,159660.80,3,1,0,113931.57,1,False,False,False
3,699,39,1.0,0.00,2,0,0,93826.63,0,False,False,False
4,850,43,2.0,125510.82,1,1,1,79084.10,0,False,True,False


#### 2.2.2 Label Encoding

In [16]:
cat_columns = ['Geography', 'Gender']

encoder = OrdinalEncoder()
df_cat = pd.DataFrame(
    encoder.fit_transform(df[cat_columns]), 
    columns=cat_columns
)

df_le = df
df_le[cat_columns] = df_cat
df_le.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,0.0,0.0,42,2.0,0.00,1,1,1,101348.88,1
1,608,2.0,0.0,41,1.0,83807.86,1,0,1,112542.58,0
2,502,0.0,0.0,42,8.0,159660.80,3,1,0,113931.57,1
3,699,0.0,0.0,39,1.0,0.00,2,0,0,93826.63,0
4,850,2.0,0.0,43,2.0,125510.82,1,1,1,79084.10,0


### 2.3 - Splitting

In [17]:
# split df_ohe into train (0.6), validation (0.2) and test (0.2)
df_train, df_val_test = train_test_split(df_ohe, test_size=0.4, random_state=123)
df_val, df_test = train_test_split(df_val_test, test_size=0.5, random_state=123)

features_train_ohe = df_train.drop('Exited', axis=1)
target_train_ohe = df_train['Exited']

features_val_ohe = df_val.drop('Exited', axis=1)
target_val_ohe = df_val['Exited']

features_test_ohe = df_test.drop('Exited', axis=1)
target_test_ohe = df_test['Exited']

In [18]:
# split df_le into train (0.6), validation (0.2) and test (0.2)
df_train, df_val_test = train_test_split(df_le, test_size=0.4, random_state=123)
df_val, df_test = train_test_split(df_val_test, test_size=0.5, random_state=123)

features_train_le = df_train.drop('Exited', axis=1)
target_train_le = df_train['Exited']

features_val_le = df_val.drop('Exited', axis=1)
target_val_le = df_val['Exited']

features_test_le = df_test.drop('Exited', axis=1)
target_test_le = df_test['Exited']

### 2.4 - Scaling

In [19]:
# columns to scale
cols_sca = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 
            'EstimatedSalary']

# scale df_ohe
scaler_ohe = StandardScaler()
scaler_ohe.fit(features_train_ohe[cols_sca])

features_train_ohe[cols_sca] = scaler_ohe.transform(features_train_ohe[cols_sca])
features_val_ohe[cols_sca] = scaler_ohe.transform(features_val_ohe[cols_sca])
features_test_ohe[cols_sca] = scaler_ohe.transform(features_test_ohe[cols_sca])

# scale df_le
scaler_le = StandardScaler()
scaler_le.fit(features_train_le[cols_sca])

features_train_le[cols_sca] = scaler_le.transform(features_train_le[cols_sca])
features_val_le[cols_sca] = scaler_le.transform(features_val_le[cols_sca])
features_test_le[cols_sca] = scaler_le.transform(features_test_le[cols_sca])

## 3 - Model with Imbalance
Model without correcting the target imbalance.

### 3.1 - LogisticRegression
Logistic Regression model performs better with One-hot encoding.

In [20]:
model_lr, f1_lr, roc_auc_lr = lr_model_scores(
    features_train_ohe,
    target_train_ohe,
    features_val_ohe,
    target_val_ohe
)

scores_df.loc[len(scores_df)] = ['model_lr', f1_lr, roc_auc_lr]

train score: 0.809
val score: 0.822
F1 Score: 0.328
ROC AUC: 0.768


### 3.2 - RandomForestClassifier

Random Forest model performs better with Label Encoding.

In [21]:
model_rfc, f1_rfc, roc_auc_rfc = rfc_optimizer(
    features_train_le,
    target_train_le,
    features_val_le,
    target_val_le
)

scores_df.loc[len(scores_df)] = ['model_rfc', f1_rfc, roc_auc_rfc]

best_depth: 14
best_n_estimators: 24
train score: 0.959
val score: 0.883
F1 Score: 0.623
ROC AUC: 0.850


### 3.3 - DecisionTreeClassifier

In [22]:
model_dtc, f1_dtc, roc_auc_dtc = dtc_optimizer(
    features_train_le,
    target_train_le,
    features_val_le,
    target_val_le
)

scores_df.loc[len(scores_df)] = ['model_dtc', f1_dtc, roc_auc_dtc]

best_depth: 6
train score: 0.866
val score: 0.864
F1 Score: 0.578
ROC AUC: 0.841


## 4 - Model without Imbalance
Model correcting the target imbalance.

In [23]:
# one-hot encoding with upsample
features_train_ohe_up, target_train_ohe_up = upsample(features_train_ohe,
                                                      target_train_ohe)

# label encoding with upsample
features_train_le_up, target_train_le_up = upsample(features_train_le,
                                                    target_train_le)

# one-hot encoding with downsample
features_train_ohe_down, target_train_ohe_down = downsample(features_train_ohe,
                                                            target_train_ohe)

# label encoding with downsample
features_train_le_down, target_train_le_down = downsample(features_train_le,
                                                          target_train_le)

### 4.1 LogisticRegression


#### 4.1.1 - Model Class Weight Parameter
Correcting with model class_weight='balanced parameter, that changes the weight to compensate the imbalance.

In [24]:
model_lr_we, f1_lr_we, roc_auc_lr_we = lr_model_scores(
    features_train_ohe,
    target_train_ohe,
    features_val_ohe,
    target_val_ohe,
    weight='balanced'
)

scores_df.loc[len(scores_df)] = ['model_lr_we', f1_lr_we, roc_auc_lr_we]

train score: 0.713
val score: 0.719
F1 Score: 0.481
ROC AUC: 0.771


#### 4.1.2 - Upsample

In [25]:
model_lr_up, f1_lr_up, roc_auc_lr_up = lr_model_scores(
    features_train_ohe_up,
    target_train_ohe_up,
    features_val_ohe,
    target_val_ohe,
)

scores_df.loc[len(scores_df)] = ['model_lr_up', f1_lr_up, roc_auc_lr_up]

train score: 0.714
val score: 0.714
F1 Score: 0.480
ROC AUC: 0.771


#### 4.1.3 - Downsample

In [26]:
model_lr_down, f1_lr_down, roc_auc_lr_down = lr_model_scores(
    features_train_ohe_down,
    target_train_ohe_down,
    features_val_ohe,
    target_val_ohe,
)

scores_df.loc[len(scores_df)] = ['model_lr_down', f1_lr_down, roc_auc_lr_down]

train score: 0.713
val score: 0.722
F1 Score: 0.485
ROC AUC: 0.771


### 4.2 - RandomForestClassifier

#### 4.2.1 - Model Class Weight Parameter

In [27]:
model_rfc_we, f1_rfc_we, roc_auc_rfc_we = rfc_optimizer(
    features_train_le,
    target_train_le,
    features_val_le,
    target_val_le,
    weight='balanced'
)

scores_df.loc[len(scores_df)] = ['model_rfc_we', f1_rfc_we, roc_auc_rfc_we]

best_depth: 7
best_n_estimators: 6
train score: 0.826
val score: 0.830
F1 Score: 0.621
ROC AUC: 0.854


#### 4.2.2 - Upsample

In [28]:
model_rfc_up, f1_rfc_up, roc_auc_rfc_up = rfc_optimizer(
    features_train_le_up,
    target_train_le_up,
    features_val_le,
    target_val_le,
)

scores_df.loc[len(scores_df)] = ['model_rfc_up', f1_rfc_up, roc_auc_rfc_up]

best_depth: 15
best_n_estimators: 29
train score: 0.994
val score: 0.855
F1 Score: 0.612
ROC AUC: 0.853


#### 4.2.3 - Downsample

In [29]:
model_rfc_down, f1_rfc_down, roc_auc_rfc_down = rfc_optimizer(
    features_train_le_down,
    target_train_le_down,
    features_val_le,
    target_val_le,
)

scores_df.loc[len(scores_df)] = ['model_rfc_down', f1_rfc_down, roc_auc_rfc_down]

best_depth: 9
best_n_estimators: 19
train score: 0.852
val score: 0.811
F1 Score: 0.601
ROC AUC: 0.855


### 4.3 - DecisionTreeClassifier

#### 4.3.1 - Model Class Weight Parameter

In [30]:
model_dtc_we, f1_dtc_we, roc_auc_dtc_we = dtc_optimizer(
    features_train_le,
    target_train_le,
    features_val_le,
    target_val_le,
    weight='balanced'
)

scores_df.loc[len(scores_df)] = ['model_dtc_we', f1_dtc_we, roc_auc_dtc_we]

best_depth: 5
train score: 0.800
val score: 0.819
F1 Score: 0.596
ROC AUC: 0.852


#### 4.3.2 - Upsample

In [31]:
model_dtc_up, f1_dtc_up, roc_auc_dtc_up = dtc_optimizer(
    features_train_le_up,
    target_train_le_up,
    features_val_le,
    target_val_le
)

scores_df.loc[len(scores_df)] = ['model_dtc_up', f1_dtc_up, roc_auc_dtc_up]

best_depth: 5
train score: 0.753
val score: 0.819
F1 Score: 0.596
ROC AUC: 0.852


#### 4.3.3 - Downsample

In [32]:
model_dtc_down, f1_dtc_down, roc_auc_dtc_down = dtc_optimizer(
    features_train_le_down,
    target_train_le_down,
    features_val_le,
    target_val_le
)

scores_df.loc[len(scores_df)] = ['model_dtc_down', f1_dtc_down, roc_auc_dtc_down]

best_depth: 5
train score: 0.766
val score: 0.820
F1 Score: 0.600
ROC AUC: 0.849


## 5 - F1 and AUC ROC Scores

In [33]:
scores_df

,model,f1,auc_roc
0,model_lr,0.327801,0.768443
1,model_rfc,0.623009,0.849631
2,model_dtc,0.577778,0.841137
3,model_lr_we,0.480652,0.770834
4,model_lr_up,0.480000,0.770832
5,model_lr_down,0.485219,0.771183
6,model_rfc_we,0.620859,0.854218
7,model_rfc_up,0.611521,0.853097
8,model_rfc_down,0.600928,0.854615
9,model_dtc_we,0.596319,0.851691


In [34]:
var_names = %who_ls
for var in var_names:
    if (var.startswith('f1') and var != 'f1_score' 
        or var.startswith('roc_auc') and var != 'roc_auc_score'):
        value = globals()[var]
        print(f'{var} = {value:.3f}')

f1_dtc = 0.578
f1_dtc_down = 0.600
f1_dtc_up = 0.596
f1_dtc_we = 0.596
f1_lr = 0.328
f1_lr_down = 0.485
f1_lr_up = 0.480
f1_lr_we = 0.481
f1_rfc = 0.623
f1_rfc_down = 0.601
f1_rfc_up = 0.612
f1_rfc_we = 0.621
roc_auc_dtc = 0.841
roc_auc_dtc_down = 0.849
roc_auc_dtc_up = 0.852
roc_auc_dtc_we = 0.852
roc_auc_lr = 0.768
roc_auc_lr_down = 0.771
roc_auc_lr_up = 0.771
roc_auc_lr_we = 0.771
roc_auc_rfc = 0.850
roc_auc_rfc_down = 0.855
roc_auc_rfc_up = 0.853
roc_auc_rfc_we = 0.854
